In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/DAT SALUD MENTAL - TAMIZAJE.csv", sep=",", encoding="latin-1")

In [ ]:
df.isnull().sum()

,0
Anio,0
NroMes,0
ubigeo,0
Departamento,0
Provincia,0
Distrito,0
Sexo,0
Etapa,0
GrupoTamizaje,0
DetalleTamizaje,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236527 entries, 0 to 236526
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   Anio             236527 non-null  int64 
 1   NroMes           236527 non-null  int64 
 2   ubigeo           236527 non-null  int64 
 3   Departamento     236527 non-null  object
 4   Provincia        236527 non-null  object
 5   Distrito         236527 non-null  object
 6   Sexo             236527 non-null  object
 7   Etapa            236527 non-null  object
 8   GrupoTamizaje    236527 non-null  object
 9   DetalleTamizaje  236527 non-null  object
 10  Casos            236527 non-null  int64 
dtypes: int64(4), object(7)
memory usage: 19.9+ MB


In [ ]:
df['Departamento'].unique()

array(['ANCASH', 'APURIMAC', 'AREQUIPA', 'AYACUCHO', 'CAJAMARCA',
       'CALLAO', 'CUSCO', 'HUANCAVELICA', 'HUANUCO', 'ICA', 'JUNIN',
       'LA LIBERTAD', 'LAMBAYEQUE', 'LIMA', 'LORETO', 'MADRE DE DIOS',
       'MOQUEGUA', 'PASCO', 'PIURA', 'PUNO', 'SAN MARTIN', 'TACNA',
       'UCAYALI'], dtype=object)

In [ ]:
df['Sexo'].unique()

array(['F', 'M'], dtype=object)

In [ ]:
object_cols = df.select_dtypes(include='object').columns

# Contar cuántas filas tienen espacios al inicio o al final por columna
espacios_extra = {}
for col in object_cols:
    # Verifica si hay espacios al principio (^\s+) o al final (\s+$)
    con_espacios = df[col].astype(str).str.contains(r'^\s+|\s+$', regex=True)
    espacios_extra[col] = con_espacios.sum()

# Mostrar el resultado
pd.Series(espacios_extra)

,0
Departamento,0
Provincia,0
Distrito,0
Sexo,0
Etapa,0
GrupoTamizaje,0
DetalleTamizaje,0


In [ ]:
print(df['Etapa'].unique())
print(df['GrupoTamizaje'].unique())
print(df['DetalleTamizaje'].unique())

['18 - 24' '40 - 59' '80  +' '30 - 39' '5 - 9' '15 - 17' '1 - 4' '60 - 79'
 '25 - 29' '10 - 11' '< 1' '12 - 14']
['TOTAL DE TAMIZAJES' 'SOLO TAMIZAJES POSITIVOS'
 'TAMIZAJES C/CONDICION ADICIONAL VIOLENCIA POLITICA']
['SINDROME Y/O TRASTORNO PSICOTICO' 'VIOLENCIA FAMILIAR/MALTRATO INFANTIL'
 'TRASTORNO DEPRESIVO' 'TRASTORNO DE CONSUMO DE ALCOHOL Y OTROS DROGAS'
 'TRASTORNO DE CONSUMO DE ALCOHOL' 'TRASTORNO DE CONSUMO DE DROGAS'
 'TRASTORNO DE CONSUMO DE TABACO' 'AFECTADO POR VIOLENCIA POLITICA']


In [ ]:
# Creacion de dim_ubicación

In [ ]:
dim_ubicacion = df[['ubigeo', 'Departamento', 'Provincia', 'Distrito']].drop_duplicates().reset_index(drop=True)
dim_ubicacion.index += 1  # ID_Ubicacion
dim_ubicacion.index.name = 'ID_Ubicacion'

In [ ]:
# Creacion de dim_población

In [ ]:
dim_poblacion = df[['Sexo', 'Etapa']].drop_duplicates().reset_index(drop=True)
dim_poblacion.index += 1  # ID_Poblacion
dim_poblacion.index.name = 'ID_Poblacion'

In [ ]:
# Creacion de dim_tamizaje

In [ ]:
dim_tamizaje = df[['GrupoTamizaje', 'DetalleTamizaje']].drop_duplicates().reset_index(drop=True)
dim_tamizaje.index += 1  # ID_Tamizaje
dim_tamizaje.index.name = 'ID_Tamizaje'

In [ ]:
# Creacion de dim_tiempo

In [ ]:
dim_tiempo = df[['Anio', 'NroMes']].drop_duplicates().reset_index(drop=True)
dim_tiempo.rename(columns={'NroMes': 'Mes'}, inplace=True)
fechas = pd.to_datetime(dim_tiempo['Anio'].astype(str) + '-' + dim_tiempo['Mes'].astype(str) + '-01')
dim_tiempo['NombreMes'] = fechas.dt.strftime('%B')
dim_tiempo['Trimestre'] = fechas.dt.quarter
dim_tiempo['Semestre'] = dim_tiempo['Mes'].apply(lambda x: 1 if x <= 6 else 2)
dim_tiempo.index += 1  # ID_Tiempo
dim_tiempo.index.name = 'ID_Tiempo'

In [ ]:
fact = df.copy()

fact = fact.merge(dim_ubicacion.reset_index(), left_on=['ubigeo', 'Departamento', 'Provincia', 'Distrito'], right_on=['ubigeo', 'Departamento', 'Provincia', 'Distrito'])
fact = fact.merge(dim_poblacion.reset_index(), on=['Sexo', 'Etapa'])
fact = fact.merge(dim_tamizaje.reset_index(), on=['GrupoTamizaje', 'DetalleTamizaje'])
fact = fact.merge(dim_tiempo.reset_index(), left_on=['Anio', 'NroMes'], right_on=['Anio', 'Mes'])

In [ ]:
fact_final = fact[['ID_Tiempo', 'ID_Ubicacion', 'ID_Poblacion', 'ID_Tamizaje', 'Casos']]

In [ ]:
#Exportamos los csv

In [ ]:
dim_ubicacion.to_csv('Dim_Ubicacion.csv')
dim_poblacion.to_csv('Dim_Poblacion.csv')
dim_tamizaje.to_csv('Dim_Tamizaje.csv')
dim_tiempo.to_csv('Dim_Tiempo.csv')
fact_final.to_csv('Fact_Tamizaje.csv', index=False)